# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. **Unit of Analysis:**  
   One row = one unique content item (page) for a given client, aggregated over a specific calendar month

2. **Tables To be Used**
   - `fact_content_daily_performance_sample` (for daily search and analytics signals aggregated to month grain)
   - `dim_content` (for page properties like `word_count` and `content_type` joined via `content_hash_id`)
   - `dim_clients` (for holdout splitting and client profile checks via `client_hash_id`)

3. **Time Window:**  
   A single calendar month: **June 1, 2026 to June 30, 2026** (`month = '2026-06'`), which is the latest complete month partition available in the sample dataset.

4. **What Will be Predicted or Ranked :**  
  The score and rank pages by their **position-adjusted CTR gap** (defined as `tier_median_ctr - observed_ctr` where the median is computed within the page's position tier). This serves as a ranking proxy for click opportunity. A positive gap tells us a page performs worse than its ranking peers.

5. **Deliberately Excluded:**  
   - `is_declining_label` / `trend_direction` / `trend_pct` for the target month: these are trailing outcomes and would leak target signals.
   - Client IDs and Content IDs as model features: they are pseudonym strings used strictly for joins and client-holdout validation splits.
   - Future GSC/GA4 records: no data from after June 30, 2026 is allowed to enter the features or targets.

In [9]:
# Load environment variables and cache Hugging Face parquet datasets locally for fast, reliable execution
import duckdb
import os, getpass
from dotenv import load_dotenv, find_dotenv
from huggingface_hub import hf_hub_download

# Search parent directories up to project root for .env
load_dotenv(find_dotenv(usecwd=True), override=True)
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF_TOKEN')

REPO_ID = 'FlyRank/internship-warehouse'
print('Downloading/loading cached parquet files from Hugging Face...')
dim_clients_path = hf_hub_download(repo_id=REPO_ID, filename='dim_clients.parquet', repo_type='dataset', token=HF_TOKEN).replace('\\', '/')
dim_content_path = hf_hub_download(repo_id=REPO_ID, filename='dim_content.parquet', repo_type='dataset', token=HF_TOKEN).replace('\\', '/')
fact_sample_path = hf_hub_download(repo_id=REPO_ID, filename='fact_content_daily_performance_sample.parquet', repo_type='dataset', token=HF_TOKEN).replace('\\', '/')

con = duckdb.connect()
TABLES = {
    'dim_clients': f"read_parquet('{dim_clients_path}')",
    'dim_content': f"read_parquet('{dim_content_path}')",
    'fact_daily_sample': f"read_parquet('{fact_sample_path}')",
}
print('DuckDB connected and local tables configured successfully.')


Downloading/loading cached parquet files from Hugging Face...
DuckDB connected and local tables configured successfully.


## 2. Fields: feature / label / context / excluded

| Field Name | Label | Context | Knowable at Decision Moment? |
| --- | --- | --- | --- |
| `total_impressions` | **Feature** | Sum of GSC impressions | **Yes:** Fully observed over the target month. |
| `avg_position` | **Feature** | Weighted avg position in GSC | **Yes:** Aggregated from recorded queries in GSC. |
| `engagement_rate` | **Feature** | GA4 engaged sessions / total sessions | **Yes:** Analytics sessions are logged before review. |
| `word_count` | **Feature** | Page word count from `dim_content` | **Yes:** Current page length is fully measurable. |
| `position_tier` | **Feature** | Derived bucket from `avg_position` | **Yes:** Computed from observed position records. |
| `ctr_gap` | **Label / Proxy** | `tier_median_ctr - observed_ctr` | **No:** This is our target opportunity metric. |
| `is_opportunity` | **Label / Proxy** | Binary target (`ctr_gap > 0` and `impressions >= 1000`) | **No:** Evaluated post-hoc to calculate Precision@K. |
| `client_hash_id` | **Context** | Grouping and holdout splitting | **Yes:** Used for splitting, not as a model feature. |
| `content_hash_id` | **Context** | Unique identifier for pages | **Yes:** Used for joins, not as a model feature. |
| `observed_ctr` | **Context / Trap** | Raw CTR (`clicks / impressions * 100`) | **No (Excluded):** Directly encodes target outcome. |
| `trend_direction` | **Excluded** | Product trend flags | **No:** Post-hoc trend outcome, would leak information. |
| `trend_pct` | **Excluded** | Raw percentage trend | **No:** Label-derived field, excluded to prevent leakage. |

In [10]:
# Proof of five features - Build feature frame query
feature_query = f"""
SELECT 
    f.client_hash_id,
    f.content_hash_id,
    SUM(f.gsc_impressions) AS total_impressions,
    CASE WHEN SUM(f.gsc_impressions) > 0 
         THEN CAST(SUM(f.gsc_clicks) AS DOUBLE) / SUM(f.gsc_impressions) * 100.0 
         ELSE 0.0 
    END AS observed_ctr,
    CASE WHEN SUM(f.gsc_impressions) > 0 
         THEN SUM(f.gsc_avg_position * f.gsc_impressions) / SUM(f.gsc_impressions) 
         ELSE 0.0 
    END AS avg_position,
    CASE WHEN SUM(f.ga4_sessions) > 0 
         THEN CAST(SUM(f.ga4_engaged_sessions) AS DOUBLE) / SUM(f.ga4_sessions) * 100.0 
         ELSE 0.0 
    END AS engagement_rate,
    ANY_VALUE(d.word_count) AS word_count
FROM {TABLES['fact_daily_sample']} f
LEFT JOIN {TABLES['dim_content']} d ON f.content_hash_id = d.content_hash_id
WHERE f.month = '2026-06'
GROUP BY f.client_hash_id, f.content_hash_id
HAVING total_impressions >= 500 AND avg_position > 0
LIMIT 5
"""
con.sql(feature_query).df()

,client_hash_id,content_hash_id,total_impressions,observed_ctr,avg_position,engagement_rate,word_count
0,client_f623b01661d4bfe4,content_7b50820e76b1a006,1636.0,0.305623,73.806846,0.0,1122
1,client_d211cb07b9059bab,content_e0d36b792d778382,930.0,0.430108,9.940860,20.0,1746
2,client_23a62021009f63c4,content_db7dec213bc5bdaf,1112.0,0.000000,44.459532,0.0,5145
3,client_23a62021009f63c4,content_1b86d6e570c74c6d,1506.0,0.199203,7.583665,5.0,3104
4,client_23a62021009f63c4,content_80c9b98274c79f85,685.0,0.291971,13.224818,0.0,6716


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


- 1: **The Grain**: Prove that grouping by client and content hash ID over our month window results in unique rows (zero rows with count > 1).
- 2: **Row Counts & Date Span**: Show the unique pages, minimum date, and maximum date for `month = '2026-06'`.
- 3: **Availability**: Show how many daily records survive the `gsc_data_available` and `ga4_data_available` flags being `TRUE`.

In [11]:
print("-- Fact 1: Grain Check --")
grain_q = f"""
WITH monthly_agg AS (
    SELECT client_hash_id, content_hash_id
    FROM {TABLES['fact_daily_sample']}
    WHERE month = '2026-06'
    GROUP BY client_hash_id, content_hash_id
)
SELECT client_hash_id, content_hash_id, COUNT(*) AS count
FROM monthly_agg
GROUP BY client_hash_id, content_hash_id
HAVING count > 1
LIMIT 5
"""
df_grain = con.sql(grain_q).df()
print(f"Rows with duplicate grain: {len(df_grain)}")

print("\n-- Fact 2: Row Count & Date Span --")
span_q = f"""
SELECT COUNT(DISTINCT content_hash_id) AS unique_pages,
       MIN(report_date) AS min_date,
       MAX(report_date) AS max_date
FROM {TABLES['fact_daily_sample']}
WHERE month = '2026-06'
"""
print(con.sql(span_q).df().to_string(index=False))

print("\n-- Fact 3: Availability Breakdown --")
avail_q = f"""
SELECT 
    COUNT(*) AS total_daily_rows,
    SUM(CASE WHEN gsc_data_available = TRUE THEN 1 ELSE 0 END) AS gsc_available,
    SUM(CASE WHEN ga4_data_available = TRUE THEN 1 ELSE 0 END) AS ga4_available,
    SUM(CASE WHEN gsc_data_available = TRUE AND ga4_data_available = TRUE THEN 1 ELSE 0 END) AS both_available
FROM {TABLES['fact_daily_sample']}
WHERE month = '2026-06'
"""
print(con.sql(avail_q).df().to_string(index=False))

-- Fact 1: Grain Check --
Rows with duplicate grain: 0

-- Fact 2: Row Count & Date Span --
 unique_pages   min_date   max_date
       409205 2026-06-01 2026-06-30

-- Fact 3: Availability Breakdown --
 total_daily_rows  gsc_available  ga4_available  both_available
         11694072      3878937.0       644726.0        554216.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Limits of the Dataset

1. **Unbalanced history:** Different clients have varying history limits (`gsc_data_start`). A global calendar threshold will fail; time windows must align on a client-level basis.
2. **GSC-only early history:** Many clients lack GA4 records prior to `ga4_data_start`. Using `ga4_pageviews` or `sessions` without filtering on `ga4_data_available = TRUE` will make older records look like they had 0 engagement, which is false.
3. **No causality:** The data is observational only. A page being flagged with a large CTR gap does not guarantee that updating its title or description will cause a CTR recovery. We can only claim priority recommendations, never causal outcomes.

---

### The Trap: Leakage Evaluation 
Demonstrate how adding a single label-derived feature (`observed_ctr`) destroys the integrity of the evaluation. 

- **Honest Features:** `total_impressions`, `avg_position`, `engagement_rate`, `word_count`
- **Leaked Feature:** Adding `observed_ctr` (since the opportunity label is computed directly from whether a page's CTR is below the tier median, this feature leaks the answer).

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# Load full aggregated dataset for June 2026
data_q = f"""
WITH monthly_agg_raw AS (
    SELECT 
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions) AS total_impressions,
        CASE WHEN SUM(f.gsc_impressions) > 0 
             THEN CAST(SUM(f.gsc_clicks) AS DOUBLE) / SUM(f.gsc_impressions) * 100.0 
             ELSE 0.0 
        END AS observed_ctr,
        CASE WHEN SUM(f.gsc_impressions) > 0 
             THEN SUM(f.gsc_avg_position * f.gsc_impressions) / SUM(f.gsc_impressions) 
             ELSE 0.0 
        END AS avg_position,
        CASE WHEN SUM(f.ga4_sessions) > 0 
             THEN CAST(SUM(f.ga4_engaged_sessions) AS DOUBLE) / SUM(f.ga4_sessions) * 100.0 
             ELSE 0.0 
        END AS engagement_rate,
        ANY_VALUE(d.word_count) AS word_count
    FROM {TABLES['fact_daily_sample']} f
    LEFT JOIN {TABLES['dim_content']} d ON f.content_hash_id = d.content_hash_id
    WHERE f.month = '2026-06'
    GROUP BY f.client_hash_id, f.content_hash_id
    HAVING total_impressions >= 500 AND avg_position > 0
),
monthly_agg AS (
    SELECT m.*,
        CASE 
            WHEN m.avg_position = 0 THEN 'no_data'
            WHEN m.avg_position <= 3 THEN 'top_3'
            WHEN m.avg_position <= 10 THEN 'page_1'
            WHEN m.avg_position <= 20 THEN 'striking'
            WHEN m.avg_position <= 50 THEN 'page_3_5'
            ELSE 'deep'
        END AS position_tier
    FROM monthly_agg_raw m
)
SELECT m.*, t.tier_median_ctr
FROM monthly_agg m
LEFT JOIN (
    SELECT position_tier, MEDIAN(observed_ctr) AS tier_median_ctr
    FROM monthly_agg
    GROUP BY position_tier
) t ON m.position_tier = t.position_tier
"""
df = con.sql(data_q).df()
df["word_count"] = df["word_count"].fillna(0)
df["ctr_gap"] = df["tier_median_ctr"] - df["observed_ctr"]
df["is_opportunity"] = ((df["ctr_gap"] > 0) & (df["total_impressions"] >= 1000)).astype(int)

# Set up Client Holdout Validation (GroupKFold)
groups = df["client_hash_id"]
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(df, df["is_opportunity"], groups))

X_train, X_test = df.iloc[train_idx], df.iloc[test_idx]

honest_features = ["total_impressions", "avg_position", "engagement_rate", "word_count"]
leaked_features = honest_features + ["observed_ctr"]

# Evaluate Honest Model
clf_h = RandomForestClassifier(random_state=42, n_estimators=50, max_depth=6)
clf_h.fit(X_train[honest_features], X_train["is_opportunity"])
p_h = clf_h.predict_proba(X_test[honest_features])[:, 1]
auc_h = roc_auc_score(X_test["is_opportunity"], p_h)
ap_h = average_precision_score(X_test["is_opportunity"], p_h)

# Evaluate Leaked Model
clf_l = RandomForestClassifier(random_state=42, n_estimators=50, max_depth=6)
clf_l.fit(X_train[leaked_features], X_train["is_opportunity"])
p_l = clf_l.predict_proba(X_test[leaked_features])[:, 1]
auc_l = roc_auc_score(X_test["is_opportunity"], p_l)
ap_l = average_precision_score(X_test["is_opportunity"], p_l)

print("-- Trap Evaluation --")
print(f"Honest Model (no observed_ctr): ROC AUC = {auc_h:.4f}, Average Precision = {ap_h:.4f}")
print(f"Leaked Model (with observed_ctr): ROC AUC = {auc_l:.4f}, Average Precision = {ap_l:.4f}")

-- Trap Evaluation --
Honest Model (no observed_ctr): ROC AUC = 0.8995, Average Precision = 0.8680
Leaked Model (with observed_ctr): ROC AUC = 1.0000, Average Precision = 1.0000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.